<a href="https://colab.research.google.com/github/fatihemreakardere/Calendar-Generator/blob/main/finetuning/notebooks/lfm3_cpt_with_trl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Continued Pre-Training (CPT) of LFM3-1.7B with TRL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/finetuning/notebooks/lfm3_cpt_with_trl.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-Liquid4All%2Fcookbook-181717?logo=github)](https://github.com/Liquid4All/cookbook) [![Discord](https://img.shields.io/discord/1385439864920739850?color=7289da&label=Discord&logo=discord&logoColor=white)](https://discord.com/invite/liquid-ai)

This notebook continues the pre-training of [LFM3-1.7B](https://huggingface.co/LiquidAI/LFM3-1.7B) on raw text from a new domain, using [TRL](https://github.com/huggingface/trl) and LoRA. It runs on a free Colab T4 GPU, or any GPU with at least 16 GB of memory.

Continued pre-training (CPT) teaches a model the vocabulary, style, and knowledge of a domain from unlabeled text, with the same next-token prediction objective as pre-training. Our domain is **Turkish**: LFM3-1.7B officially supports 16 languages, and Turkish is not one of them. We train on Turkish Wikipedia articles and measure how well the model predicts held-out Turkish text before and after.

## 🎯 What you'll find

- **Raw text data**: stream a slice of a large corpus from the Hugging Face Hub
- **LoRA + CPT**: train on plain text packed into full rows, as in pre-training
- **Evaluation**: perplexity on held-out articles and on out-of-distribution web pages, and a Turkish grammar test, before and after training
- **Export**: merge the LoRA weights into a standalone checkpoint

# 📦 Installation & Setup

In [1]:
!pip install -qqq "torch==2.14.1" "torchvision==0.29.1" "torchao==0.18.0" "trl==1.16.0" "transformers==5.20.2" "peft==0.21.2" "accelerate==1.15.0" "datasets==5.1.0" "huggingface-hub==2.3.0" "tokenizers==0.23.3" "safetensors==0.8.0" --progress-bar off

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [2]:
import math
import os
import torch
import transformers
import trl
import peft

os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"📊 TRL version: {trl.__version__}")
print(f"🎛️ PEFT version: {peft.__version__}")

# LFM3-1.7B is released in bfloat16. GPUs without bfloat16 support, like the T4, load it in float16 instead:
# the conversion is exact for almost every weight, and the LoRA adapters train in float32 either way.
use_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"🔢 Precision: {dtype}")

W1029 19:02:49.419000 530 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1029 19:02:49.514000 530 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


📦 PyTorch version: 2.14.1+cu130
🤗 Transformers version: 5.20.2
📊 TRL version: 1.16.0
🎛️ PEFT version: 0.21.2
🔢 Precision: torch.float16


# Loading the model from Transformers 🤗

We continue training the post-trained LFM3-1.7B, which keeps its chat abilities for the instruction tuning that usually follows CPT. For large CPT runs, consider starting from the base checkpoint instead.

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "LiquidAI/LFM3-1.7B"

print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

print("🧠 Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)

print("✅ Model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")

📚 Loading tokenizer...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:140: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

configuration_lfm3.py:   0%|          | 0.00/6.11k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/LiquidAI/LFM3-1.7B:
- configuration_lfm3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.46GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/286 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

✅ Model loaded successfully!
🔢 Parameters: 1,728,801,536


# 📚 Load a domain corpus

[Wikipedia](https://huggingface.co/datasets/wikimedia/wikipedia) has one subset per language, and the Turkish one alone has more than 500,000 articles. We stream it, so that only the articles we use are downloaded: 3,000 for training and 100 for evaluation. To check that the model learns Turkish and not just the style of Wikipedia, we also evaluate on 100 Turkish web pages from [FineWeb-2](https://huggingface.co/datasets/HuggingFaceFW/fineweb-2) (news, forums, shops), which the model never sees during training.

Each article becomes one `text` example, which is the format TRL expects for plain language modeling. We add the end-of-sequence token so that the model learns where documents end.

In [4]:
from datasets import Dataset, load_dataset

stream = load_dataset("wikimedia/wikipedia", "20231101.tr", split="train", streaming=True).shuffle(seed=42, buffer_size=10_000)

def to_text(article):
    return {"text": f"{article['title']}\n\n{article['text']}{tokenizer.eos_token}"}

# Skip stub articles, which are mostly lists and templates
articles = (to_text(article) for article in stream if len(article["text"]) > 2_000)
eval_dataset = Dataset.from_list([next(articles) for _ in range(100)])
train_dataset = Dataset.from_list([next(articles) for _ in range(3_000)])

# Out-of-distribution check: Turkish web pages (news, forums, shops), a different register from Wikipedia
web = load_dataset("HuggingFaceFW/fineweb-2", name="tur_Latn", split="test", streaming=True)
ood_dataset = Dataset.from_list([{"text": page["text"] + tokenizer.eos_token} for page in web.take(100)])

n_tokens = sum(len(ids) for ids in tokenizer(list(train_dataset["text"]))["input_ids"])
print(f"📚 Train articles: {len(train_dataset)} ({n_tokens / 1e6:.1f}M tokens)")
print(f"🧪 Eval articles: {len(eval_dataset)}")
print(f"🌐 Out-of-distribution web pages: {len(ood_dataset)}")
print(f"\n📝 Sample:\n{train_dataset[0]['text'][:500]}")

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/329k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

📚 Train articles: 3000 (10.7M tokens)
🧪 Eval articles: 100
🌐 Out-of-distribution web pages: 100

📝 Sample:
Sıkıyönetim

Sıkıyönetim veya askerî adalet; askerî otoritenin, genellikle resmî bildirgesi altında, adli yönetimi kontrol altına almasıyla işleme geçen kural sistemidir.

Askerî adalet; ortaya çıkan savaş, doğal afet, sivil kargaşa, toprak işgali veya askerî darbe gibi durumlarda, normal adli kurumların yeni duruma hâkim olamayıp vazifesini yerine getiremediği veya yavaş getirdiği takdirde, askerî otoritelerin ve kurumların tercih edilmesi ile kullanıma girer.

Sıkıyönetim; tabii afet, salgın h


# 🔍 A grammar test: Turkish minimal pairs

Perplexity tells us that the model predicts Turkish text better. To check that it learned Turkish grammar, and not just the vocabulary of Wikipedia, we use [TurBLiMP](https://huggingface.co/datasets/juletxara/turblimp), a benchmark of minimal pairs: two sentences that differ in one detail, where only one is grammatical. A model "passes" a pair when it gives a higher probability to the grammatical sentence. The pairs cover 16 phenomena (agreement, binding, word order, and more), and none of them come from Wikipedia.

We use 100 pairs per phenomenon, and score the model before training.

In [5]:
from datasets import concatenate_datasets

phenomena = ["anaphor_agreement", "argument_structure_ditransitive", "argument_structure_transitive", "binding", "determiners", "ellipsis", "irregular_forms", "island_effects", "nominalization", "npi_licensing", "passives", "quantifiers", "relative_clauses", "scrambling", "subject_agreement", "suspended_affixation"]
pairs = concatenate_datasets([load_dataset("juletxara/turblimp", p, split="train").shuffle(seed=0).select(range(100)) for p in phenomena])

@torch.no_grad()
def sentence_logprobs(model, sentences, batch_size=64):
    """Total log-probability of each sentence."""
    model.eval()
    tokenizer.padding_side = "right"
    scores = []
    for i in range(0, len(sentences), batch_size):
        inputs = tokenizer([tokenizer.bos_token + s for s in sentences[i:i + batch_size]], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        logits = model(**inputs, use_cache=False).logits.float()
        token_logprobs = torch.log_softmax(logits[:, :-1], dim=-1).gather(-1, inputs["input_ids"][:, 1:, None])[..., 0]
        scores += (token_logprobs * inputs["attention_mask"][:, 1:]).sum(-1).tolist()
    return scores

def grammar_test(model):
    """For each pair, whether the model prefers the grammatical sentence."""
    good = sentence_logprobs(model, list(pairs["sentence_good"]))
    bad = sentence_logprobs(model, list(pairs["sentence_bad"]))
    return [g > b for g, b in zip(good, bad)]

def by_phenomenon(passed):
    return {p: sum(ok for ok, q in zip(passed, pairs["linguistic_phenomenon"]) if q == p) / sum(q == p for q in pairs["linguistic_phenomenon"]) for p in sorted(set(pairs["linguistic_phenomenon"]))}

original_passed = grammar_test(model)
print(f"📏 TurBLiMP accuracy before training: {sum(original_passed) / len(pairs):.1%}")

README.md:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

data/anaphor_agreement-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B / 45.3kB            

data/anaphor_agreement-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

data/argument_structure_ditransitive-000(…): reconstructing file:   0%|          |  0.00B / 41.3kB            

...

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

data/suspended_affixation-00000-of-00001(…): reconstructing file:   0%|          |  0.00B / 39.8kB            

data/suspended_affixation-00000-of-00001(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

📏 TurBLiMP accuracy before training: 69.5%


# 🎯 Continued Pre-Training (CPT + LoRA)

## Wrap the model with PEFT

For CPT we use a higher LoRA rank than for SFT, because the model has more to learn. We adapt the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`), the gated short convolutions (`in_proj`, `out_proj`), and the MLP (`w1`, `w2`, `w3`). LFM3's attention layers also have a per-head output gate, `gate_proj`; adding it to the targets is a reasonable choice.

In [6]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=64,
    lora_alpha=64,
    lora_dropout=0.0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", "in_proj", "out_proj", "w1", "w2", "w3",
        # "gate_proj",  # optional: LFM3's per-head attention gate
    ],
    task_type="CAUSAL_LM",
)

## Configure training

The dataset has a `text` column, so `SFTTrainer` trains on every token, like pre-training. We also enable packing, as in pre-training: `packing_strategy="wrapped"` concatenates all the articles, separated by the end-of-sequence token, and cuts the stream into rows of exactly 2,048 tokens. No compute is wasted on padding, and long articles are split across rows instead of truncated.

In [7]:
from trl import SFTConfig, SFTTrainer

cpt_config = SFTConfig(
    output_dir="./LFM3-1.7B-CPT-tr",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # <- 8 rows of 2,048 tokens per step
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,  # <- a float is a fraction of the total steps
    max_length=2048,
    packing=True,
    packing_strategy="wrapped",
    dataset_text_field="text",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    per_device_eval_batch_size=2,
    save_strategy="no",
    bf16=use_bf16,
    fp16=not use_bf16,  # <- float16 mixed precision with loss scaling on GPUs without bfloat16
    report_to="none",
)

print("🏗️  Creating CPT trainer...")
trainer = SFTTrainer(
    model=model,
    args=cpt_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()

🏗️  Creating CPT trainer...


Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

trainable params: 64,421,888 || all params: 1,793,223,424 || trainable%: 3.5925


## Evaluate the model before training

Perplexity is the exponential of the average loss per token: lower means the model predicts the held-out Turkish text better.

In [8]:
baseline = trainer.evaluate()
baseline_ood = trainer.evaluate(eval_dataset=ood_dataset, metric_key_prefix="ood")
print(f"📏 Perplexity before training: {math.exp(baseline['eval_loss']):.2f} (Wikipedia), {math.exp(baseline_ood['ood_loss']):.2f} (web pages)")

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,3.245184,0,2.995653,0.432116,0.000000


Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,4.091238,0,3.662124,0.332172,0.000000


📏 Perplexity before training: 25.67 (Wikipedia), 59.81 (web pages)


## Launch training

In [ ]:
print("🚀 Starting continued pre-training...")
trainer.train()
print("🎉 Continued pre-training completed!")
print(f"🧮 Peak GPU memory: {torch.cuda.max_memory_reserved() / 1e9:.1f} GB")

final = trainer.evaluate()
final_ood = trainer.evaluate(eval_dataset=ood_dataset, metric_key_prefix="ood")
print(f"📏 Perplexity before training: {math.exp(baseline['eval_loss']):.2f} (Wikipedia), {math.exp(baseline_ood['ood_loss']):.2f} (web pages)")
print(f"📏 Perplexity after training:  {math.exp(final['eval_loss']):.2f} (Wikipedia), {math.exp(final_ood['ood_loss']):.2f} (web pages)")

🚀 Starting continued pre-training...


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
100,2.582651,2.519215,2.835589,0.510011,1638400.000000
200,2.451678,2.435790,2.684623,0.522057,3276800.000000
300,2.453007,2.386819,2.649551,0.528430,4913721.000000


# 🔍 Original vs fine-tuned model

We run the grammar test again with the trained model, and look at pairs where training changed the model's choice.

In [ ]:
finetuned_passed = grammar_test(trainer.model)
original_scores, finetuned_scores = by_phenomenon(original_passed), by_phenomenon(finetuned_passed)

# Pairs the original model got wrong and the fine-tuned model gets right, in the phenomena that improved most
most_improved = sorted(phenomena, key=lambda p: finetuned_scores[p] - original_scores[p], reverse=True)[:4]
for p in most_improved:
    i = next(i for i, (o, f, q) in enumerate(zip(original_passed, finetuned_passed, pairs["linguistic_phenomenon"])) if q == p and not o and f)
    print(f"\n🧩 {p.replace('_', ' ')}: {original_scores[p]:.0%} -> {finetuned_scores[p]:.0%}")
    print(f"   ✅ grammatical:   {pairs[i]['sentence_good']}")
    print(f"   ❌ ungrammatical: {pairs[i]['sentence_bad']}")
    print(f"   Original prefers the ungrammatical one, the fine-tuned model prefers the grammatical one.")

And the summary:

In [ ]:
print(f"{'':<34}{'original':>10}{'fine-tuned':>12}")
print(f"{'perplexity, Wikipedia (held out)':<34}{math.exp(baseline['eval_loss']):>10.2f}{math.exp(final['eval_loss']):>12.2f}")
print(f"{'perplexity, web pages (OOD)':<34}{math.exp(baseline_ood['ood_loss']):>10.2f}{math.exp(final_ood['ood_loss']):>12.2f}")
print(f"{'TurBLiMP grammar accuracy':<34}{sum(original_passed) / len(pairs):>10.1%}{sum(finetuned_passed) / len(pairs):>12.1%}")
for p in phenomena:
    print(f"{'  ' + p.replace('_', ' '):<34}{original_scores[p]:>10.0%}{finetuned_scores[p]:>12.0%}")

# 💾 Save and merge the model

We save the LoRA adapter, then merge it into the base weights. `save_pretrained` also copies LFM3's modeling code next to the weights, so the merged model loads with `trust_remote_code=True` like the original one.

In [ ]:
trainer.save_model("./LFM3-1.7B-CPT-tr-lora")
print("💾 LoRA adapter saved to: ./LFM3-1.7B-CPT-tr-lora")

output_dir = "./LFM3-1.7B-CPT-tr-merged"
print("🔄 Merging LoRA weights...")
merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"💾 Merged model saved to: {output_dir}")

# To push the merged model to the Hugging Face Hub, uncomment:
# merged_model.push_to_hub("your-username/LFM3-1.7B-CPT-tr")
# tokenizer.push_to_hub("your-username/LFM3-1.7B-CPT-tr")

# 🚀 Next steps

- **Use more data**: CPT benefits from far more text than this demo. Increase the number of articles, or stream your own corpus with `load_dataset("json", data_files=..., streaming=True)`.
- **Teach the model to follow instructions in the new domain**: after CPT, run a short SFT on conversations from that domain (see [`lfm3_sft_with_trl.ipynb`](./lfm3_sft_with_trl.ipynb)), starting from the merged checkpoint.
- **Train the embeddings too**: for a new language with many unseen words, also training the input embeddings helps.